# R(s) vs R_final on P1-P6 Prompts
Compute per-token features (ce_var, agree_rate, coherence) for all 8 humanization prompts.
This validates that R_final works across different prompt strategies, not just P5E.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1+cu121', 'torchvision==0.20.1+cu121', '--index-url', 'https://download.pytorch.org/whl/cu121', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==4.46.3', 'accelerate==1.1.1', 'bitsandbytes', 'scipy', 'scikit-learn', '-q'])
import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, compute {p.major}.{p.minor}')
    x = torch.randn(2,2).cuda(); print(f'OK: {(x@x).sum().item():.2f}'); del x

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 98.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 108.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.6 MB/s eta 0:00:00
PyTorch 2.5.1+cu121, CUDA: True
GPU: Tesla P100-PCIE-16GB, compute 6.0
OK: 1.87


In [2]:
import gc, os, sys, json, shutil, warnings
import numpy as np
import torch, torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
BASE = '/kaggle/input'
df = ds = None
for r, dirs, files in os.walk(BASE):
    for f in files:
        if f == 'humanization_dataset_v2.json': df = os.path.join(r, f)
        if f == 'metrics.py' and 'dna' not in os.path.basename(r): ds = r
    for d in dirs:
        if d == 'dna_detectllm':
            c = os.path.join(r, d)
            if os.path.isfile(os.path.join(c, '__init__.py')): ds = c
if ds and not os.path.exists('dna_detectllm'):
    os.makedirs('dna_detectllm', exist_ok=True)
    for f in os.listdir(ds):
        if f.endswith('.py'): shutil.copy(os.path.join(ds, f), 'dna_detectllm/')
    if not os.path.exists('dna_detectllm/__init__.py'): open('dna_detectllm/__init__.py','w').close()
sys.path.insert(0, os.getcwd())
from dna_detectllm.metrics import sum_perplexity, entropy
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
with open(df) as f: dataset = json.load(f)
print(f'Human: {len(dataset["human_texts"])}')
for k in dataset['humanized']:
    n = sum(1 for x in dataset['humanized'][k] if x)
    label = dataset['metadata']['prompt_labels'].get(k, k)
    print(f'  {label}: {n} valid')

Human: 100
  Simple Paraphrase: 100 valid
  Burstiness & Structure: 99 valid
  Personal Voice & Opinions: 100 valid
  Conversational Tone: 99 valid
  Anti-AI Lexicon: 100 valid
  Kitchen Sink (All Combined): 99 valid
  PA: Vivid Lexicon + Structure: 100 valid
  PB: Sharp Journalist Persona: 100 valid


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import ctypes
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect(); torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained('tiiuae/falcon-7b')
tokenizer.pad_token = tokenizer.eos_token
MAX_LEN = 256; DEVICE = 'cuda:0'
cc = torch.cuda.get_device_properties(0)
if (cc.major, cc.minor) >= (7,0):
    from transformers import BitsAndBytesConfig
    qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    lk = dict(quantization_config=qc, device_map='auto', low_cpu_mem_usage=True); MAX_LEN = 512
else:
    lk = dict(torch_dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True)
observer = AutoModelForCausalLM.from_pretrained('tiiuae/falcon-7b', **lk); observer.eval()
gc.collect(); torch.cuda.empty_cache()
try: ctypes.CDLL('libc.so.6').malloc_trim(0)
except: pass
performer = AutoModelForCausalLM.from_pretrained('tiiuae/falcon-7b-instruct', **lk); performer.eval()
print(f'Models loaded. MAX_LEN={MAX_LEN}')

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2026-04-15 21:13:50.164882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776287630.362694      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776287630.415978      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776287630.863326      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776287630.863367      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776287630.863370      23 computation_placer.cc:177] computation placer alr

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Models loaded. MAX_LEN=256


In [4]:
@torch.inference_mode()
def compute_all(text):
    torch.cuda.empty_cache()
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN, return_token_type_ids=False)
    eg = {k: v.to(DEVICE) for k, v in enc.items()}
    ol = observer(**eg).logits.float()
    pl = performer(**eg).logits.float()
    class E:
        def __init__(s, d): s.input_ids = d['input_ids']; s.attention_mask = d['attention_mask']
        def to(s, dev): s.input_ids = s.input_ids.to(dev); s.attention_mask = s.attention_mask.to(dev); return s
    eo = E(eg)
    ppl = sum_perplexity(eo, pl); xppl = entropy(ol, pl, eo, tokenizer.pad_token_id)
    rs = float(ppl[0] / (2 * xppl[0]))
    shifted = pl[..., :-1, :]; labels = eg['input_ids'][..., 1:]; attn = eg['attention_mask'][..., 1:]
    ce = F.cross_entropy(shifted.transpose(1,2), labels, reduction='none')
    S = ce[attn.bool()].cpu().numpy()
    if len(S) < 5: del ol, pl, eg; return None
    ce_var = float(np.var(S))
    obs_top = ol[..., :-1, :].argmax(dim=-1); perf_top = shifted.argmax(dim=-1)
    agree = (obs_top == perf_top).float()[attn.bool()].cpu().numpy()
    agree_rate = float(np.mean(agree))
    pe = -(F.softmax(pl[..., :-1, :], dim=-1) * F.log_softmax(pl[..., :-1, :], dim=-1)).sum(-1)
    H = pe[attn.bool()].cpu().numpy(); H_med = np.median(H); conf = H < H_med
    coherence = float(np.mean(S[conf]) - np.mean(S[~conf])) if conf.sum() > 0 and (~conf).sum() > 0 else 0.0
    del ol, pl, eg
    return {'rs': rs, 'ce_var': ce_var, 'agree_rate': agree_rate, 'coherence': coherence}
print(f'Test: {compute_all("Hello world test.")}')

Test: None


In [5]:
# Score all groups
CKPT = 'p1p6_ckpt.json'
if os.path.exists(CKPT):
    all_sc = json.load(open(CKPT)); print(f'Checkpoint: {list(all_sc.keys())}')
else:
    all_sc = {}
groups = {'human': dataset['human_texts'], 'original_ai': dataset['original_ai_texts']}
for k in dataset['humanized']:
    groups[k] = dataset['humanized'][k]
for gn, texts in groups.items():
    if gn in all_sc: print(f'{gn}: done'); continue
    print(f'Scoring {gn}...')
    sc = []
    for t in tqdm(texts, desc=gn):
        if t is None: sc.append(None)
        else:
            try: sc.append(compute_all(t))
            except Exception as e: print(f'Err: {e}'); sc.append(None)
    all_sc[gn] = sc
    with open(CKPT, 'w') as f: json.dump(all_sc, f)
print('All scored!')

Scoring human...


human: 100%|██████████| 100/100 [07:02<00:00,  4.22s/it]


Scoring original_ai...


original_ai: 100%|██████████| 100/100 [07:13<00:00,  4.33s/it]


Scoring P1_simple_paraphrase...


...

Scoring PA_vivid_lexicon...


PA_vivid_lexicon: 100%|██████████| 100/100 [07:12<00:00,  4.33s/it]


Scoring PB_journalist_persona...


PB_journalist_persona: 100%|██████████| 100/100 [07:13<00:00,  4.33s/it]

All scored!


In [6]:
# Compute R_final for all prompts
hv = [s for s in all_sc.get('human', []) if s]
h_cev = [s['ce_var'] for s in hv]; h_agr = [s['agree_rate'] for s in hv]; h_coh = [s['coherence'] for s in hv]
CM, CS = np.mean(h_cev), np.std(h_cev)
AM, AS = np.mean(h_agr), np.std(h_agr)
OM, OS = np.mean(h_coh), np.std(h_coh)
print(f'Human baseline: ce_var={CM:.2f}+/-{CS:.2f}, agree={AM:.4f}+/-{AS:.4f}, coh={OM:.2f}+/-{OS:.2f}')

def R_final(s, k=1.0):
    rs = s['rs']
    p1 = max(0.1, 1 - 0.1 * max(0, s['ce_var'] - (CM + k * CS)))
    p2 = max(0.1, 1 - 3.0 * max(0, (AM - k * AS) - s['agree_rate']))
    p3 = max(0.1, 1 - 0.5 * max(0, (OM - k * OS) - s['coherence']))
    return rs * p1 * p2 * p3

h_rs = [s['rs'] for s in hv]
h_rf = [R_final(s) for s in hv]

print(f'\n{"Prompt":<35} {"R(s) AUROC":>12} {"R_final AUROC":>14} {"Delta":>8}')
print('='*72)

labels = dataset['metadata'].get('prompt_labels', {})
results = {}
for gn in all_sc:
    if gn in ('human',): continue
    gv = [s for s in all_sc[gn] if s]
    if not gv: continue
    g_rs = [s['rs'] for s in gv]
    g_rf = [R_final(s) for s in gv]
    a_rs = roc_auc_score([1]*len(h_rs)+[0]*len(g_rs), h_rs+g_rs)
    a_rf = roc_auc_score([1]*len(h_rf)+[0]*len(g_rf), h_rf+g_rf)
    label = labels.get(gn, gn)[:33]
    delta = a_rf - a_rs
    print(f'{label:<35} {a_rs:>12.4f} {a_rf:>14.4f} {delta:>+8.4f}')
    results[gn] = {'label': label, 'rs_auroc': a_rs, 'rf_auroc': a_rf}

Human baseline: ce_var=6.24+/-1.31, agree=0.7390+/-0.0663, coh=-2.41+/-0.38

Prompt                                R(s) AUROC  R_final AUROC    Delta
original_ai                               0.9916         0.9428  -0.0488
Simple Paraphrase                         0.9543         0.8834  -0.0709
Burstiness & Structure                    0.7341         0.7684  +0.0342
Personal Voice & Opinions                 0.9129         0.8804  -0.0325
Conversational Tone                       0.8824         0.7839  -0.0985
Anti-AI Lexicon                           0.6947         0.5839  -0.1108
Kitchen Sink (All Combined)               0.8257         0.8075  -0.0182
PA: Vivid Lexicon + Structure             0.7204         0.9782  +0.2578
PB: Sharp Journalist Persona              0.9094         0.8770  -0.0324


In [7]:
with open('p1p6_rfinal_results.json', 'w') as f:
    json.dump({'human_baseline': {'cev_m': CM, 'cev_s': CS, 'agr_m': AM, 'agr_s': AS, 'coh_m': OM, 'coh_s': OS}, 'results': results, 'all_scores': {k: [s for s in v if s] for k, v in all_sc.items()}}, f, indent=2)
print('Saved p1p6_rfinal_results.json')

Saved p1p6_rfinal_results.json
